# 02 — MetaOrchestrator

**Goal:** give a goal in plain English, and let agents be created on the fly to solve it.

This is what happens, step by step:

1. You give a goal, like *"design a CLI tool that converts CSV to JSON"*.
2. A **planner** agent breaks the goal into smaller subtasks.
3. For each subtask, the orchestrator finds or **spawns** a fresh agent.
4. Each agent runs in order. The results are collected.
5. The agents that worked are saved in **memory**, so the next run can reuse them.

That last point is the interesting one: the system learns which agents to spawn over time.

**Before you start:** you need a `.env` at the repo root with `LLM_API_KEY` and `LLM_MODEL`.


## 1. Load the config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. Build the planner

The planner is a normal `BaseAgent`. The only special thing about it is its **output type**: `TaskDecomposition`.

A `TaskDecomposition` contains:

- the original goal,
- a list of `SubTask`s (each with a short name, a description, and a flag saying it needs its own agent),
- a short note on the reasoning.

That's it. The planner reads the goal and returns this structured plan.


In [2]:
from orqest.agents import BaseAgent, GlobalState
from orqest.autonomy.meta import TaskDecomposition


class PlannerAgent(BaseAgent[GlobalState, TaskDecomposition]):
    def __init__(self, model, api_key=None):
        super().__init__(
            agent_name="planner",
            system_prompt=(
                "You decompose a high-level goal into 2-4 concrete subtasks. "
                "Each subtask needs a short snake_case name, a one-sentence "
                "description, and requires_agent=True. Keep the list minimal."
            ),
            output_type=TaskDecomposition,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> TaskDecomposition:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


planner = PlannerAgent(model=config.llm_model, api_key=config.llm_api_key)
print("Planner ready.")

Planner ready.


## 3. The three supporting pieces

We need three more objects:

- **`ToolRegistry`** — a shared place where tools live. Spawned agents look up tools here. For this demo we leave it empty: our agents just reason in text.
- **`AgentFactory`** — takes an `AgentSpec` (a description of what an agent should look like) and builds a real running agent from it.
- **`MetaOrchestrator`** — the conductor. It connects the planner, the factory, and the registry. `max_subtasks=4` keeps things small.


In [3]:
from orqest.autonomy import ToolRegistry, AgentFactory, MetaOrchestrator

registry = ToolRegistry()

factory = AgentFactory(
    registry,
    default_model=config.llm_model,
    api_key=config.llm_api_key,
)

meta = MetaOrchestrator(planner, factory, registry, max_subtasks=4)
print("Orchestrator ready.")

Orchestrator ready.


## 4. Solve a goal

`solve()` runs the whole loop:

1. The planner decomposes the goal.
2. For each subtask, an agent is found in memory or spawned fresh.
3. The agents run in order.
4. The results are collected into an `ExecutionResult`.

We give it a small, concrete goal so the run finishes quickly.


In [4]:
result = await meta.solve(
    "Design a command-line tool that converts CSV files to JSON: name its "
    "subcommands, list the global flags, and write one usage example."
)

print(f"goal succeeded: {result.success}")
print(f"duration: {result.total_duration_ms:.0f} ms")
print(result.summary)

goal succeeded: True
duration: 11598 ms
Goal: Design a command-line tool that converts CSV files to JSON: name its subcommands, list the global flags, and write one usage example.
  [OK] define_subcommands (via define_subcommands)
  [OK] specify_global_flags (via specify_global_flags)
  [OK] provide_usage_example (via provide_usage_example)


### Look at each subtask result

Every subtask has its own result. `was_spawned=True` means the agent was newly created during this run (not pulled from memory).

In [5]:
for r in result.subtask_results:
    status = "ok" if r.success else "FAILED"
    print(f"[{status}] {r.subtask_name}  (agent={r.agent_used}, spawned={r.was_spawned})")
    if r.success:
        print(f"        {str(r.output)[:140]}")

[ok] define_subcommands  (agent=define_subcommands, spawned=True)
        result='Here is a concise set of CLI subcommands covering common CSV to JSON conversion workflows:\n\n1. convert\n   • Description: Convert 
[ok] specify_global_flags  (agent=specify_global_flags, spawned=True)
        result='Here is a structured list of global flags commonly found in CLI tools for CSV to JSON workflows. These flags apply across all subcom
[ok] provide_usage_example  (agent=provide_usage_example, spawned=True)
        result="Example:\n\n    csv2json convert data.csv --output result.json --pretty --delimiter ';'\n\nBreakdown:\n  - 'csv2json': Name of the C


## 5. What got spawned?

The orchestrator keeps every agent it created during the run, in a dict called `spawned_agents`. The key is the subtask name.

So if a later subtask in the **same run** needs the same kind of work, the orchestrator can reuse the agent without spawning a new one.


In [6]:
for name, ag in meta.spawned_agents.items():
    print(f"{name:30s} -> {type(ag).__name__}")

define_subcommands             -> DynamicAgent
specify_global_flags           -> DynamicAgent
provide_usage_example          -> DynamicAgent


## 6. Reuse across runs — with memory

The `spawned_agents` dict above lives in memory **for one run only**. When the run ends, it is gone.

To reuse agents in a **later** run, we pass a `MemoryStore` to the orchestrator. Now when an agent is spawned, its spec is also saved to the store. A different orchestrator that shares the same store can pull the spec back from memory instead of spawning a new agent.

There is one practical problem: a real LLM planner gives slightly different subtask names every run, so the recall path looks like it never matches. To make the reuse easy to see, we use a **fixed planner** that always returns the same two subtasks.


### A fixed planner — same plan, every time

This planner does not call the LLM at all. It just returns a hardcoded `TaskDecomposition`. That way both runs produce the same subtask names, and we can clearly see memory reuse working.


In [7]:
import tempfile, os
from orqest.memory import LocalMemoryStore
from orqest.autonomy.meta import SubTask

# Memory lives in a temp directory so it doesn't pollute the repo.
store = LocalMemoryStore(os.path.join(tempfile.mkdtemp(), "meta_memory.db"))


class FixedPlanner(BaseAgent[GlobalState, TaskDecomposition]):
    """Returns the same decomposition every time. No LLM call."""

    def __init__(self):
        super().__init__(
            agent_name="fixed_planner",
            system_prompt="plan",
            output_type=TaskDecomposition,
            model=config.llm_model,
            api_key=config.llm_api_key,
        )

    async def _run_implementation(self, state, **kwargs) -> TaskDecomposition:
        return TaskDecomposition(
            goal="ship a small CLI tool",
            subtasks=[
                SubTask(
                    name="design_cli",
                    description="Design the CLI surface and arguments.",
                    requires_agent=True,
                ),
                SubTask(
                    name="write_readme",
                    description="Draft a concise README for the tool.",
                    requires_agent=True,
                ),
            ],
            reasoning="Design, then document.",
        )


print("Fixed planner and memory store ready.")

Fixed planner and memory store ready.


### Run twice — watch the store

We make a fresh orchestrator, run the goal once, and check how many entries are in the store. Then we make a **different** orchestrator with the same store and run the same goal again.

The key signal: **the store does not grow on the second run**. That tells us the agents were pulled from memory, not spawned from scratch.


In [8]:
# Run 1 — fresh orchestrator. It spawns both agents and saves them.
meta_a = MetaOrchestrator(FixedPlanner(), factory, registry, memory=store)
await meta_a.solve("ship a small CLI tool")
count_after_run_1 = await store.count()
print(f"after run 1: store holds {count_after_run_1} entries")

# Run 2 — a DIFFERENT orchestrator, same store.
# Its own in-process cache is empty, so it must rehydrate from memory.
meta_b = MetaOrchestrator(FixedPlanner(), factory, registry, memory=store)
await meta_b.solve("ship a small CLI tool")
count_after_run_2 = await store.count()

print(f"after run 2: store holds {count_after_run_2} entries", end="  ")
if count_after_run_2 == count_after_run_1:
    print("(stable -> run 2 reused the specs from memory)")
else:
    print("(grew -> no reuse)")

after run 1: store holds 4 entries
after run 2: store holds 4 entries  (stable -> run 2 reused the specs from memory)


### What got saved in memory?

The orchestrator saves each spawned agent twice: once as an **episodic** entry (a log of what happened) and once as a **procedural skill** (something to be reused). The procedural skill's `trigger` is the subtask name, which is how recall finds it on the next run.


In [9]:
skills = await store.list_recent(memory_type="procedural")
for s in skills:
    spec = s.structured_content.get("spec", {}) if s.structured_content else {}
    print(f"skill trigger={s.content!r}  ->  spec.name={spec.get('name')!r}")

skill trigger='write_readme'  ->  spec.name='write_readme'
skill trigger='design_cli'  ->  spec.name='design_cli'


## Recap

Here is the whole picture in plain words:

1. `solve(goal)` runs the planner once to get a `TaskDecomposition`.
2. For each subtask in order, the orchestrator looks for an existing agent:
   - First in its own in-process cache.
   - Then in **procedural memory** (skills keyed on the subtask name).
   - Then in **episodic memory**.
   - If nothing matches, it spawns a fresh agent.
3. A newly spawned agent's spec is saved to memory in both forms.
4. Each subtask runs through the standard hook lifecycle. So you can plug in watchdogs, event publishers, and so on — same as any other agent.

**One bonus feature:** if you pass a `MetacognitionConfig`, a low-confidence subtask result triggers **re-decomposition** of the remaining subtasks. The orchestrator re-plans in the middle of the run. (See notebook 01 for the confidence machinery.)

**One technical note:** the factory builds each agent's output type from a JSON Schema at runtime, using `pydantic.create_model`. That is why an `AgentSpec` is fully serialisable — and why an LLM can emit one as structured output.
